<a href="https://colab.research.google.com/github/brunosalme/brunosalme/blob/main/dental-model-training-j1w63v/notebooks/train_fase3_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brunosalme/PANseg/blob/claude/dental-model-training-j1w63v/notebooks/train_fase3_baseline.ipynb)

# train_fase3_baseline — YOLO11l detecção, Fase 3 (15 classes)

**M1** do plano (`planning/plan_fase3_estruturas.md`) — treino do baseline de detecção sobre o dataset combinado (`pano-fcjgf` + Dentxpert + Tooth Instance Segmentation).

**Dataset**: `PANseg/data/pano_merged_yolo/pano_merged.yaml` — 15.299 imagens (train 12.582 / valid 1.812 / test 905), 15 classes, PHI removido, sem vazamento entre splits (M0 completo).

**Meta**: mAP50 ≥ 0,80 (baselines públicos das fontes: pano-fcjgf 63,7% / Dentxpert 47,3% / Tooth Instance Segmentation 81,3%)

**Por que detecção (bbox), não segmentação**: o dataset é bbox-only nas 3 fontes — refinamento para máscara via SAM é a Etapa B, feita depois (ver §4.2 do plano), condicional ao resultado deste baseline.

**GPU obrigatória** (A100 recomendado — mesmo padrão da Fase 1/2).

## 1. Montar Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Instalar dependências

In [2]:
%pip install -q ultralytics wandb
import ultralytics
ultralytics.checks()

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Setup complete ✅ (12 CPUs, 167.1 GB RAM, 46.7/235.7 GB disk)


## 3. Verificar GPU

In [3]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU não detectada. Vá em Runtime → Change runtime type → GPU (A100 recomendado).')

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1e9
print(f'GPU  : {gpu.name}')
print(f'VRAM : {vram_gb:.1f} GB')

if vram_gb >= 75:      # A100 80GB
    batch_hint = 64
elif vram_gb >= 35:    # A100 40GB
    batch_hint = 32
elif vram_gb >= 20:    # T4/V100
    batch_hint = 12
else:
    batch_hint = 6
print(f'Batch sugerido (AutoBatch ajusta na prática): {batch_hint}')

GPU  : NVIDIA A100-SXM4-80GB
VRAM : 85.1 GB
Batch sugerido (AutoBatch ajusta na prática): 64


## 4. Configuração

In [4]:
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive')
DRIVE_DATA   = DRIVE_ROOT / 'PANseg/data/pano_merged_yolo'
DRIVE_YAML   = DRIVE_DATA / 'pano_merged.yaml'
WEIGHTS_DIR  = DRIVE_ROOT / 'PANseg/weights/fase3'

LOCAL_DATA   = Path('/content/pano_merged_yolo')
LOCAL_YAML   = LOCAL_DATA / 'pano_merged.yaml'

MODEL_NAME   = 'yolo11l.pt'     # detecção — não -seg, dataset é bbox-only
IMGSZ        = 1280
EPOCHS       = 150
PATIENCE     = 50
BATCH        = -1               # AutoBatch (Ultralytics ajusta pela VRAM disponível)
WORKERS      = 8
SEED         = 42

# projeto LOCAL do Ultralytics (rápido, evita escrever a cada época direto no Drive) —
# os pesos finais são copiados para WEIGHTS_DIR (Drive) só ao final, na célula 12
PROJECT_NAME = '/content/runs/fase3'
RUN_NAME     = 'yolo11l_det_fase3_merged'

USE_WANDB = True
WANDB_PROJECT = 'panseg-fase3'   # nome limpo — evita usar o path local como nome de projeto

## 5. Copiar dataset para local (velocidade)

Drive montado tem I/O aleatório lento (~0,4 MB/s); copiar para `/content/` local acelera o treino significativamente (mesma lição da Fase 1/2). ~15.299 imagens, deve levar alguns minutos.

In [5]:
import shutil, time

if LOCAL_DATA.exists():
    print('Já copiado localmente, pulando.')
else:
    t0 = time.time()
    shutil.copytree(str(DRIVE_DATA), str(LOCAL_DATA))
    print(f'Copiado em {(time.time() - t0)/60:.1f} min')

for split in ['train', 'valid', 'test']:
    n = len(list((LOCAL_DATA / split / 'images').glob('*')))
    print(f'{split}: {n} imagens')

Copiado em 13.0 min
train: 12582 imagens
valid: 1812 imagens
test: 905 imagens


## 6. Ajustar dataset.yaml para os caminhos locais

In [6]:
import yaml

with open(DRIVE_YAML) as f:
    yaml_data = yaml.safe_load(f)

yaml_data['path'] = str(LOCAL_DATA)

with open(LOCAL_YAML, 'w') as f:
    yaml.safe_dump(yaml_data, f, sort_keys=False, allow_unicode=True)

print(open(LOCAL_YAML).read())

path: /content/pano_merged_yolo
train: train/images
val: valid/images
test: test/images
names:
  0: Caries
  1: Crown
  2: Crown - bridge
  3: Deep Caries
  4: Filling
  5: Implant
  6: Mandibular Canal
  7: Missing teeth
  8: Periapical lesion
  9: Post-screw
  10: Retained root
  11: Root Canal Treatment
  12: Root Piece
  13: impacted tooth
  14: maxillary sinus



## 7. W&B — login + inicialização explícita do run

**Importante**: chamamos `wandb.init()` nós mesmos aqui, em vez de deixar o Ultralytics
detectar/criar o run sozinho. Isso garante um nome de projeto limpo (`panseg-fase3`) — se
deixássemos por conta do Ultralytics, ele usaria o `project` passado para `model.train()`
(um path local tipo `/content/runs/fase3`) como nome do projeto no W&B, o que ficaria
estranho/pode nem ser um nome válido. Com o run já aberto antes do `model.train()`, o
Ultralytics detecta o run existente e loga as métricas de cada época nele automaticamente.

In [7]:
wandb_run = None

if USE_WANDB:
    try:
        from google.colab import userdata
        wandb_key = userdata.get('WANDB_KEY')
    except Exception:
        wandb_key = None

    import wandb
    if wandb_key:
        wandb.login(key=wandb_key)
        print('W&B login via Colab Secrets ✅')
    else:
        wandb.login()
        print('W&B login interativo ✅')

    # habilita explicitamente a integração nativa do Ultralytics com W&B
    # (por padrão pode vir desativada nas settings internas do pacote)
    from ultralytics import settings as ul_settings
    ul_settings.update({'wandb': True})

    wandb_run = wandb.init(
        project = WANDB_PROJECT,
        name    = RUN_NAME,
        config  = {
            'model': MODEL_NAME, 'imgsz': IMGSZ, 'epochs': EPOCHS,
            'patience': PATIENCE, 'dataset': 'pano_merged (15 classes, 15299 imgs)',
        },
    )
    print(f'\nW&B run aberto: {wandb_run.url}')
    print('Acompanhe pelo link acima (funciona no navegador do celular também).')
else:
    import os
    os.environ['WANDB_DISABLED'] = 'true'
    print('W&B desativado.')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: brunosalme (brunosalme-mycomp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login interativo ✅



W&B run aberto: https://wandb.ai/brunosalme-mycomp/panseg-fase3/runs/eb8yqayw
Acompanhe pelo link acima (funciona no navegador do celular também).


## 8. Treinar

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)

results = model.train(
    data        = str(LOCAL_YAML),
    imgsz       = IMGSZ,
    epochs      = EPOCHS,
    batch       = BATCH,
    patience    = PATIENCE,
    workers     = WORKERS,
    seed        = SEED,
    project     = PROJECT_NAME,
    name        = RUN_NAME,
    save        = True,
    save_period = 10,
    val         = True,
    plots       = True,
    optimizer   = 'AdamW',
    lr0         = 0.001,
    cos_lr      = True,
    warmup_epochs = 5,
    weight_decay  = 0.0005,
    # augmentations conservadoras — OPG é grayscale, não inverter verticalmente
    hsv_h = 0.0, hsv_s = 0.0, hsv_v = 0.4,
    degrees = 5.0, translate = 0.1, scale = 0.2,
    fliplr = 0.5, flipud = 0.0,
    mosaic = 0.0, mixup = 0.0,
)

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/pano_merged_yolo/pano_merged.yaml, degrees=5.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolo11l_det_fase3_merged, n

## 9. Resultados do treino (val set)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

run_dir = Path(PROJECT_NAME) / RUN_NAME

metrics = results.results_dict
print('=== Métricas no val set (melhor época) ===')
for k, v in metrics.items():
    print(f'  {k:40s}: {v:.4f}')

plots = ['results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg']
fig, axes = plt.subplots(1, len(plots), figsize=(20, 6))
for ax, p in zip(axes, plots):
    img_path = run_dir / p
    if img_path.exists():
        ax.imshow(mpimg.imread(img_path))
        ax.set_title(p)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 10. Avaliação no test set — geral e por classe

In [ ]:
from ultralytics import YOLO

def find_best_pt():
    try:
        p = Path(results.save_dir) / 'weights' / 'best.pt'
        if p.exists():
            return p
    except NameError:
        pass
    p = run_dir / 'weights' / 'best.pt'
    if p.exists():
        return p
    raise FileNotFoundError('best.pt não encontrado — cheque o run_dir.')

best_pt = find_best_pt()
print('Usando pesos:', best_pt)
model_best = YOLO(best_pt)

test_metrics = model_best.val(
    data  = str(LOCAL_YAML),
    split = 'test',
    imgsz = IMGSZ,
    plots = True,
)

print('\n=== mAP50 por classe (teste interno) ===')
class_names = model_best.names
for i, ap50 in enumerate(test_metrics.box.ap50):
    print(f'  {class_names[i]:25s}: {ap50:.4f}')

print(f'\nmAP50 geral   : {test_metrics.box.map50:.4f}')
print(f'mAP50-95 geral: {test_metrics.box.map:.4f}')

**Comparação com baselines públicos das fontes**: pano-fcjgf 63,7% · Dentxpert 47,3% · Tooth Instance Segmentation 81,3%. Como o test set combina principalmente pano-fcjgf e Dentxpert (ver §M0d.4 do plano — tooth_instance_seg perdeu quase todo o test por causa da correção de vazamento), a comparação mais direta é com esses dois. Atenção especial às classes que **só** existem numa fonte: `Deep Caries`/`Impacted tooth`/`Periapical lesion` (Dentxpert), `Crown - bridge`/`Post-screw`/`Filling`/`Root Canal Treatment`/`Implant` (tooth_instance_seg, avaliadas majoritariamente via generalização do treino, não teste isolado).

## 11. Recall por classe — atenção às classes de risco (Retained root)

In [ ]:
print('=== Recall por classe (teste interno) ===')
for i, r in enumerate(test_metrics.box.r):
    flag = ' ⚠️ classe de baixo volume — avaliar com cautela' if class_names[i] in ('Retained root',) else ''
    print(f'  {class_names[i]:25s}: {r:.4f}{flag}')

## 12. Salvar pesos no Drive

In [ ]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
dst_dir   = WEIGHTS_DIR / f'{RUN_NAME}_{timestamp}'
dst_dir.mkdir(parents=True, exist_ok=True)

src_weights = run_dir / 'weights'
shutil.copy2(src_weights / 'best.pt', dst_dir / 'best.pt')
shutil.copy2(src_weights / 'last.pt', dst_dir / 'last.pt')
shutil.copy2(LOCAL_YAML, dst_dir / 'pano_merged.yaml')

for p in ['results.png', 'results.csv', 'confusion_matrix_normalized.png']:
    src = run_dir / p
    if src.exists():
        shutil.copy2(src, dst_dir / p)

print('Pesos e artefatos salvos em:', dst_dir)

In [ ]:
if wandb_run is not None:
    wandb_run.finish()
    print('W&B run fechado.')

## Próximo passo

Registrar os resultados em `planning/plan_fase3_estruturas.md` (mAP50/mAP50-95 geral e por classe, comparação com os baselines públicos). Se mAP50 ≥ 0,80 e nenhuma classe crítica ficar muito abaixo da média, seguir para a **Etapa B** (§4.2 do plano): refinamento de máscara via SAM guiado por bbox, validado por grupo de risco (materiais/anatomia/patologia).